In [ ]:
from langchain_core.tools import tool, InjectedToolArg
import requests
from typing import Annotated
from dotenv import load_dotenv
import os
load_dotenv()

EXCHANGE_RATE_API = os.getenv('EXCHANGE_RATE_API')


@tool
def find_exchange_rate(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency
    """
    url = f'https://v6.exchangerate-api.com/v6/{EXCHANGE_RATE_API}/pair/{base_currency}/{target_currency}'
    response = requests.get(url)
    data = response.json()
    
    return data

@tool
def converted_currency(amt: float, exchange_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    given a exchange rate this function calculates the target currency value from a given base currency value
    """
    return amt * exchange_rate

In [39]:
find_exchange_rate.invoke({'base_currency' : 'USD', 'target_currency' : 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1770768001,
 'time_last_update_utc': 'Wed, 11 Feb 2026 00:00:01 +0000',
 'time_next_update_unix': 1770854401,
 'time_next_update_utc': 'Thu, 12 Feb 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 90.5947}

In [40]:
converted_currency.invoke({'amt' : 100, 'exchange_rate' : 90.5947})

9059.470000000001

In [41]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')

model = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite', api_key=api_key)

model_w_tools = model.bind_tools([find_exchange_rate, converted_currency])

chat = []
query = HumanMessage('hi how are you')
chat.append(query)

response = model_w_tools.invoke(chat)
response.content

'I am doing well, how can I help you today?'

In [42]:
query = HumanMessage('What is the conversion factor between INR and USD, and based on that conversion rate, can you convert 10 inr to usd')
chat.append(query)
response = model_w_tools.invoke(chat)
chat.append(response)
response

AIMessage(content='', additional_kwargs={'function_call': {'name': 'converted_currency', 'arguments': '{"amt": 10}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c4b5f-ce9e-7020-b6cf-30fd799abbcb-0', tool_calls=[{'name': 'find_exchange_rate', 'args': {'target_currency': 'USD', 'base_currency': 'INR'}, 'id': '0c92830a-7f8d-408a-b7ae-b120becb72f7', 'type': 'tool_call'}, {'name': 'converted_currency', 'args': {'amt': 10}, 'id': 'e9cad67a-ada5-4c45-98d3-3e9750e85880', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 151, 'output_tokens': 42, 'total_tokens': 193, 'input_token_details': {'cache_read': 0}})

In [43]:
response.tool_calls

[{'name': 'find_exchange_rate',
  'args': {'target_currency': 'USD', 'base_currency': 'INR'},
  'id': '0c92830a-7f8d-408a-b7ae-b120becb72f7',
  'type': 'tool_call'},
 {'name': 'converted_currency',
  'args': {'amt': 10},
  'id': 'e9cad67a-ada5-4c45-98d3-3e9750e85880',
  'type': 'tool_call'}]

In [44]:
# here converted_currency fn takes the output frm find_exchange_rate
# so the input that ts fn takes is InjectedToolArg input

In [45]:
import json
xchng_rate = 0
for tool_call in response.tool_calls:
    if tool_call['name'] == 'find_exchange_rate':
        tool_response = find_exchange_rate.invoke(tool_call)    # tool_call sent for ToolMessage
        print(tool_response.content)
        xchng_rate = json.loads(tool_response.content)['conversion_rate']
        # print(xchng_rate)
        chat.append(tool_response)
        
    if tool_call['name'] == 'converted_currency':
        tool_call['args']['exchange_rate'] = xchng_rate
        # print(tool_call['args']['exchange_rate'])
        tool_response = converted_currency.invoke(tool_call)
        # print(tool_response)
        chat.append(tool_response)        

{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1770768001, "time_last_update_utc": "Wed, 11 Feb 2026 00:00:01 +0000", "time_next_update_unix": 1770854401, "time_next_update_utc": "Thu, 12 Feb 2026 00:00:01 +0000", "base_code": "INR", "target_code": "USD", "conversion_rate": 0.01104}


In [46]:
chat

[HumanMessage(content='hi how are you', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the conversion factor between INR and USD, and based on that conversion rate, can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'converted_currency', 'arguments': '{"amt": 10}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c4b5f-ce9e-7020-b6cf-30fd799abbcb-0', tool_calls=[{'name': 'find_exchange_rate', 'args': {'target_currency': 'USD', 'base_currency': 'INR'}, 'id': '0c92830a-7f8d-408a-b7ae-b120becb72f7', 'type': 'tool_call'}, {'name': 'converted_currency', 'args': {'amt': 10, 'exchange_rate': 0.01104}, 'id': 'e9cad67a-ada5-4c45-98d3-3e9750e85880', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 151, 'output_tokens': 42, 'total_tokens': 1

In [47]:
response = model_w_tools.invoke(chat)
chat.append(response)
print(response.content)

The conversion rate from INR to USD is 0.01104. Therefore, 10 INR is equal to 0.1104 USD.
